# XGBoost Solar Power Prediction

Train an **XGBoost regression model** to predict `power_now_w` from time and weather features.

- **System:** 3.57 kW rooftop, Karnal, Haryana
- **Data:** 13 months of 15-min readings (Apr 2025 – May 2026)
- **Features:** `hour`, `month`, `day_of_year`, `irradiance`, `ambient_temp`, `module_temp`
- **Target:** `power_now_w`
- **Split:** chronological 80/20 (no shuffle)

In [ ]:
!pip install xgboost shap plotly influxdb-client --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.dpi'] = 150
print('Libraries loaded.')

In [ ]:
%run data_loader.py
df = load_solar_data()
print(f'Loaded {len(df):,} rows')
df.head()

In [ ]:
# ── Feature preparation ────────────────────────────────────────────────────────
FEATURES = ['hour', 'month', 'day_of_year',
            'irradiance', 'ambient_temp', 'module_temp']
TARGET   = 'power_now_w'

df_model = df[FEATURES + [TARGET, 'datetime']].dropna().copy()
df_model['datetime'] = pd.to_datetime(df_model['datetime'])
df_model = df_model.sort_values('datetime').reset_index(drop=True)

X = df_model[FEATURES].values
y = df_model[TARGET].values

# Chronological 80/20 split — no shuffle
split_idx = int(len(df_model) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Preserve hour labels for scatter colouring
hour_test = df_model['hour'].values[split_idx:]

print(f'Train shape: {X_train.shape}  |  Test shape: {X_test.shape}')
print(f'Training period ends at: {df_model["datetime"].iloc[split_idx - 1]}')

In [ ]:
# ── Train XGBoost ──────────────────────────────────────────────────────────────
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist',
    verbosity=1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50,
    early_stopping_rounds=30
)

print(f'\nBest iteration: {model.best_iteration}')

In [ ]:
# ── Evaluation metrics ─────────────────────────────────────────────────────────
y_pred = model.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

print('=' * 42)
print(f'  R²    : {r2:.4f}')
print(f'  RMSE  : {rmse:.2f} W')
print(f'  MAE   : {mae:.2f} W')
print('=' * 42)

In [ ]:
# ── Figure: Actual vs Predicted  +  Residual Distribution ─────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('XGBoost Solar Power Prediction Results', fontsize=14, fontweight='bold')

# ── Left: scatter actual vs predicted, coloured by hour ───────────────────────
sc = ax1.scatter(
    y_test, y_pred,
    c=hour_test, cmap='viridis',
    alpha=0.4, s=10
)
plt.colorbar(sc, ax=ax1, label='Hour of Day')

lim = max(y_test.max(), y_pred.max()) * 1.02
ax1.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect prediction')
ax1.set_xlim(0, lim)
ax1.set_ylim(0, lim)
ax1.set_xlabel('Actual Power (W)')
ax1.set_ylabel('Predicted Power (W)')
ax1.set_title('Actual vs Predicted Solar Output', fontsize=12, fontweight='bold')
ax1.legend(
    title=f'R² = {r2:.4f}\nRMSE = {rmse:.1f} W',
    fontsize=9
)
ax1.grid(alpha=0.3)

# ── Right: residual histogram + KDE ───────────────────────────────────────────
residuals = y_test - y_pred

ax2.hist(
    residuals, bins=60,
    color='steelblue', edgecolor='white',
    alpha=0.85, density=True, label='Residuals'
)
ax2.axvline(0, color='black', linewidth=1.5, label='Zero line')

# KDE overlay
kde_x = np.linspace(residuals.min(), residuals.max(), 500)
kde   = gaussian_kde(residuals)
ax2.plot(kde_x, kde(kde_x), color='darkorange', linewidth=2, label='KDE')

ax2.set_xlabel('Residual (W)')
ax2.set_ylabel('Density')
ax2.set_title(
    'Residual Distribution (should be ~normal, centred at 0)',
    fontsize=12, fontweight='bold'
)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('xgboost_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: xgboost_results.png')

In [ ]:
# ── Save model ─────────────────────────────────────────────────────────────────
model.save_model('solar_xgb_model.json')
print('Model saved to: solar_xgb_model.json')
print('Load it later with: model = XGBRegressor(); model.load_model("solar_xgb_model.json")')

## Model Summary

| Hyperparameter | Value |
|---|---|
| `n_estimators` | 500 (with early stopping) |
| `learning_rate` | 0.05 |
| `max_depth` | 6 |
| `subsample` | 0.8 |
| `colsample_bytree` | 0.8 |

**Interpreting the residual plot:**  
A near-normal distribution centred at 0 indicates well-calibrated predictions with no systematic bias.  
Heavy tails or a skewed distribution would suggest the model under/over-predicts at extreme output levels (e.g. very cloudy or very clear days).

**Next step:** Run `03_shap_analysis.ipynb` to understand which features drive individual predictions.